In [44]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
##DATA LOADING
df = pd.read_csv("Battery_Data_Cleaned.csv")
df.head()


In [ ]:
df.describe()

In [ ]:
df.info()

### Statistik Deskriptif untuk Fitur Kunci

Kita akan menganalisis distribusi statistik untuk `Capacity`, `Re`, dan `Rct` untuk membantu menentukan ambang batas klasifikasi.

In [ ]:
print("Statistik Deskriptif untuk Capacity:")
display(df['Capacity'].describe())

print("\nStatistik Deskriptif untuk Re (Resistance):")
display(df['Re'].describe())

print("\nStatistik Deskriptif untuk Rct (Resistance to Charge Transfer):")
display(df['Rct'].describe())

In [ ]:
print("Statistik Deskriptif untuk Rct (Resistance to Charge Transfer):")
display(df['Rct'].describe())

In [ ]:
df.isnull().sum()

In [ ]:
corr_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matriks Korelasi Antar Fitur")
plt.show()

# Task
Analisis data baterai, definisikan kriteria klasifikasi berdasarkan statistik deskriptif dari 'Capacity', 'Re', dan 'Rct', buat kolom status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman'), visualisasikan distribusi status baterai, dan jelaskan penggunaan fitur serta output untuk AI/front-end.

## Definisikan Kriteria Klasifikasi

### Subtask:
Berdasarkan statistik deskriptif 'Capacity', 'Re', dan 'Rct' yang telah dihitung, definisikan ambang batas (thresholds) untuk mengklasifikasikan baterai ke dalam kategori 'aman', 'perlu di test lebih lanjut', dan 'tidak aman'.


```markdown
### Menentukan Kriteria Klasifikasi Baterai

Berdasarkan statistik deskriptif yang telah kita hitung, kita dapat mulai mendefinisikan ambang batas untuk mengklasifikasikan status baterai sebagai 'aman', 'perlu di test lebih lanjut', atau 'tidak aman'. Kita akan menggunakan fitur `Capacity`, `Re`, dan `Rct`.

Berikut adalah statistik deskriptif untuk setiap fitur:

**Capacity (Kapasitas Baterai):**
- `count`: 7368
- `mean`: 0.8249
- `std`: 0.2503
- `min`: 0.0000
- `25%`: 0.7751
- `50%`: 0.8948
- `75%`: 0.9865
- `max`: 1.2920

**Re (Resistance):**
- `count`: 7368
- `mean`: 0.0777
- `std`: 0.0226
- `min`: 0.0267
- `25%`: 0.0609
- `50%`: 0.0747
- `75%`: 0.0958
- `max`: 0.1421

**Rct (Resistance to Charge Transfer):**
- `count`: 7368
- `mean`: 0.1251
- `std`: 0.0448
- `min`: 0.0388
- `25%`: 0.0847
- `50%`: 0.1184
- `75%`: 0.1589
- `max`: 0.2381

#### Proposal Ambang Batas Klasifikasi:

**1. Capacity:**
*   **Aman**: Capacity > Mean (0.8249) atau > 75th percentile (0.9865) jika kita ingin lebih konservatif. Baterai dengan kapasitas di atas rata-rata atau mendekati kapasitas penuh dianggap aman.
*   **Perlu di test lebih lanjut**: Capacity antara 25th percentile (0.7751) dan Mean (0.8249). Ini menunjukkan kapasitas yang sedikit menurun tetapi mungkin masih dapat digunakan atau memerlukan pemeriksaan lebih lanjut.
*   **Tidak Aman**: Capacity < 25th percentile (0.7751). Kapasitas yang sangat rendah menunjukkan baterai yang tidak berfungsi dengan baik atau sudah usang.

**2. Re (Resistance):**
*   **Aman**: Re < Mean (0.0777). Resistansi internal yang rendah menunjukkan kondisi baterai yang baik.
*   **Perlu di test lebih lanjut**: Re antara Mean (0.0777) dan 75th percentile (0.0958). Peningkatan resistansi ini bisa menjadi indikasi awal masalah.
*   **Tidak Aman**: Re > 75th percentile (0.0958). Resistansi internal yang tinggi seringkali dikaitkan dengan degradasi baterai dan risiko keamanan.

**3. Rct (Resistance to Charge Transfer):**
*   **Aman**: Rct < Mean (0.1251). Resistansi transfer muatan yang rendah menunjukkan efisiensi pengisian/pengosongan yang baik.
*   **Perlu di test lebih lanjut**: Rct antara Mean (0.1251) dan 75th percentile (0.1589). Peningkatan Rct bisa berarti ada masalah pada antarmuka elektroda atau elektrolit.
*   **Tidak Aman**: Rct > 75th percentile (0.1589). Rct yang tinggi menunjukkan masalah serius dalam kemampuan baterai untuk mengalirkan muatan, berpotensi berbahaya.

**Kriteria Gabungan (Contoh):**
Untuk klasifikasi akhir, kita bisa menggabungkan kriteria ini. Misalnya:
*   **Aman**: Jika `Capacity` tinggi **DAN** `Re` rendah **DAN** `Rct` rendah.
*   **Tidak Aman**: Jika `Capacity` sangat rendah **ATAU** `Re` sangat tinggi **ATAU** `Rct` sangat tinggi.
*   **Perlu di test lebih lanjut**: Kasus di antaranya, atau jika hanya satu atau dua kriteria menunjukkan anomali ringan.

Dalam langkah selanjutnya, kita akan mengimplementasikan kriteria ini ke dalam DataFrame untuk membuat kolom 'Battery_Status'.
```

## Buat Kolom Status Baterai

### Subtask:
Implementasikan logika klasifikasi yang telah didefinisikan untuk membuat kolom baru di DataFrame, misalnya 'Battery_Status', yang berisi label 'aman', 'perlu di test lebih lanjut', atau 'tidak aman'.


**Reasoning**:
I will define the thresholds based on the descriptive statistics derived in the previous subtask and then apply these rules to create the 'Battery_Status' column using `np.select`.



In [ ]:
import numpy as np

# Define thresholds based on previous analysis (using descriptive stats as guidelines)
# Capacity thresholds
capacity_25th = 0.7751
capacity_mean = 0.8249
capacity_75th = 0.9865

# Re thresholds
re_mean = 0.0777
re_75th = 0.0958

# Rct thresholds
rct_mean = 0.1251
rct_75th = 0.1589

# Define conditions for each status
conditions = [
    # Aman: Capacity tinggi DAN Re rendah DAN Rct rendah
    (df['Capacity'] > capacity_mean) & (df['Re'] < re_mean) & (df['Rct'] < rct_mean),

    # Tidak Aman: Capacity sangat rendah ATAU Re sangat tinggi ATAU Rct sangat tinggi
    (df['Capacity'] < capacity_25th) | (df['Re'] > re_75th) | (df['Rct'] > rct_75th)
]

# Define choices for each status
choices = ['aman', 'tidak aman']

# Apply np.select to create the 'Battery_Status' column
# Any case not covered by 'aman' or 'tidak aman' will be 'perlu di test lebih lanjut'
df['Battery_Status'] = np.select(conditions, choices, default='perlu di test lebih lanjut')

print("Value counts for 'Battery_Status':")
print(df['Battery_Status'].value_counts())


## Visualisasikan Distribusi Status Baterai

### Subtask:
Buat visualisasi (misalnya, diagram batang atau pie chart) untuk menunjukkan distribusi jumlah baterai di setiap kategori status (aman, perlu di test lebih lanjut, tidak aman) yang baru dibuat. Pastikan untuk menambahkan legenda pada visualisasi.


**Reasoning**:
I will create a bar plot to visualize the distribution of 'Battery_Status' using seaborn's countplot, as specified in the instructions, to clearly show the number of batteries in each category.



In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Battery_Status', hue='Battery_Status', palette='viridis', legend=False)
plt.title('Distribusi Status Baterai')
plt.xlabel('Status Baterai')
plt.ylabel('Jumlah')
plt.show()

## Jelaskan Penggunaan Fitur dan Output untuk AI/Front-end

### Subtask:
Berikan penjelasan detail tentang mengapa `Capacity`, `Re`, dan `Rct` dipilih sebagai fitur input untuk model AI, dan bagaimana output klasifikasi 'Battery_Status' akan disajikan di front-end display. Ini akan mencakup bagaimana pengguna akan memasukkan data ini dan bagaimana hasilnya akan ditampilkan.


```markdown
### Penggunaan Fitur Input dan Output untuk Aplikasi AI/Front-end

Dalam proyek ini, fitur-fitur kunci yang dipilih untuk klasifikasi status baterai adalah `Capacity`, `Re` (Resistance), dan `Rct` (Resistance to Charge Transfer). Pemilihan ini didasarkan pada signifikansi fisika dan diagnostik mereka terhadap kesehatan baterai:

*   **Capacity (Kapasitas)**: Ini adalah indikator langsung dari kemampuan baterai untuk menyimpan muatan listrik. Penurunan kapasitas seiring waktu adalah tanda utama degradasi baterai. Baterai dengan kapasitas yang lebih tinggi umumnya dianggap lebih sehat.
*   **Re (Internal Resistance)**: Resistansi internal adalah ukuran seberapa efisien baterai dapat menghantarkan arus listrik. Peningkatan resistansi internal seringkali mengindikasikan degradasi sel baterai, penuaan, atau bahkan masalah keamanan seperti _thermal runaway_.
*   **Rct (Resistance to Charge Transfer)**: Resistansi transfer muatan mengukur kemudahan ion bergerak melintasi antarmuka elektroda-elektrolit selama siklus pengisian dan pengosongan. Peningkatan Rct dapat menunjukkan masalah pada lapisan permukaan elektroda, yang memengaruhi kinerja dan efisiensi baterai.

**Mengapa Fitur Ini Penting untuk AI?**

Fitur-fitur ini secara kolektif memberikan gambaran komprehensif tentang kondisi internal baterai. Model AI, setelah dilatih, akan mempelajari pola dan ambang batas dalam kombinasi nilai-nilai `Capacity`, `Re`, dan `Rct` untuk memprediksi status kesehatan baterai. Misalnya, baterai dengan kapasitas tinggi, resistansi internal rendah, dan resistansi transfer muatan rendah kemungkinan besar akan diklasifikasikan sebagai 'aman'. Sebaliknya, kombinasi kapasitas rendah dan resistansi tinggi akan mengarah pada klasifikasi 'tidak aman'.

**Bagaimana Model AI Menggunakan Input?**

Model AI (misalnya, model klasifikasi seperti _Support Vector Machine_, _Random Forest_, atau _Neural Network_) akan menerima tiga nilai numerik ini (`Capacity`, `Re`, `Rct`) sebagai fitur input. Setiap set nilai ini akan diproses oleh model untuk menghasilkan prediksi status baterai tunggal ('aman', 'perlu di test lebih lanjut', atau 'tidak aman'). Proses ini meniru logika ambang batas yang telah kita definisikan secara manual, tetapi dengan kemampuan model AI untuk mengidentifikasi hubungan yang lebih kompleks dan non-linear antar fitur jika diperlukan.

**Penyajian Output di Front-end:**

Ketika pengguna menginteraksikan dengan sistem yang didukung AI ini (misalnya, melalui aplikasi web atau mobile), mereka akan perlu memasukkan atau melihat data baterai. Berikut adalah skenario umum:

1.  **Input Data Pengguna**: Pengguna dapat memasukkan data `Capacity`, `Re`, dan `Rct` secara manual melalui formulir input, mengunggah file CSV yang berisi beberapa data baterai, atau data dapat secara otomatis diumpankan dari sensor yang terhubung ke baterai.

2.  **Pemrosesan oleh Backend AI**: Setelah data diterima, _backend_ sistem akan mengirimkan nilai-nilai fitur ini ke model AI yang telah dilatih. Model AI kemudian akan mengklasifikasikan status baterai.

3.  **Tampilan Output Front-end**: Hasil klasifikasi (`Battery_Status`) akan dikembalikan ke _front-end_ dan ditampilkan kepada pengguna dengan cara yang intuitif dan mudah dipahami. Contoh tampilan:
    *   **Teks Sederhana**: "Status Baterai: Aman" atau "Status Baterai: Tidak Aman".
    *   **Indikator Warna**: Warna hijau untuk 'aman', kuning untuk 'perlu di test lebih lanjut', dan merah untuk 'tidak aman'.
    *   **Grafik**: Dashboard yang menampilkan tren status baterai dari waktu ke waktu atau perbandingan antara beberapa baterai.
    *   **Rekomendasi Tindakan**: Selain status, sistem juga dapat menyertakan rekomendasi tindakan, seperti "Baterai ini aman untuk digunakan", "Disarankan untuk melakukan inspeksi lebih lanjut", atau "Segera ganti baterai ini".

Dengan cara ini, pengguna, bahkan tanpa keahlian teknis mendalam tentang kimia baterai, dapat dengan cepat memahami kondisi kesehatan baterai dan mengambil keputusan yang tepat berdasarkan prediksi model AI.
```

## Final Task

### Subtask:
Menyimpulkan proses penetapan fitur dan label untuk model AI, serta bagaimana data ini akan digunakan untuk tampilan front-end.


# Task
Menganalisis data baterai, mendefinisikan kriteria klasifikasi berdasarkan statistik deskriptif dari 'Capacity', 'Re', dan 'Rct', membuat kolom status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman'), memvisualisasikan distribusi status baterai, dan menjelaskan penggunaan fitur serta output untuk AI/front-end.

## Klarifikasi Penggunaan Fitur Input

### Subtask:
Jelaskan secara spesifik fitur-fitur yang ada di DataFrame `df` saat ini, yaitu `Capacity`, `Re`, `Rct`, dan `ambient_temperature`. Jelaskan mengapa `Capacity`, `Re`, dan `Rct` dipilih sebagai fitur inti untuk klasifikasi yang telah kita buat. Kemudian, diskusikan peran potensial dari `ambient_temperature` (yang memang ada di dataset) sebagai fitur tambahan. Serta, diskusikan juga `voltage` dan `current` (yang tidak ada di DataFrame `df` saat ini), mengapa keduanya penting dalam analisis baterai secara umum, dan bagaimana fitur-fitur ini bisa menjadi input di masa depan jika data tersedia.


### Klarifikasi Penggunaan Fitur Input

Dalam analisis baterai ini, DataFrame `df` kita saat ini memiliki beberapa fitur penting:

*   **Capacity (Kapasitas)**: Mengukur seberapa banyak energi yang dapat disimpan oleh baterai. Ini adalah indikator langsung dari kesehatan dan kemampuan baterai untuk berfungsi. Kapasitas yang menurun adalah tanda utama degradasi baterai.
*   **Re (Internal Resistance)**: Merepresentasikan resistansi internal baterai. Peningkatan resistansi ini dapat menunjukkan degradasi sel, pemanasan berlebih, atau masalah internal lainnya yang mempengaruhi efisiensi dan keamanan baterai.
*   **Rct (Resistance to Charge Transfer)**: Mengukur resistansi terhadap transfer muatan ion pada antarmuka elektroda-elektrolit. Peningkatan Rct dapat menunjukkan masalah pada reaksi kimia baterai, yang mempengaruhi laju pengisian dan pengosongan serta efisiensi.
*   **ambient_temperature (Suhu Lingkungan)**: Suhu di sekitar baterai. Ini adalah faktor eksternal yang diketahui dapat memengaruhi kinerja dan umur baterai.

#### Mengapa `Capacity`, `Re`, dan `Rct` Dipilih sebagai Fitur Inti

`Capacity`, `Re`, dan `Rct` dipilih sebagai fitur inti untuk klasifikasi status baterai karena secara langsung mencerminkan kondisi fisik dan kimia internal baterai. Mereka adalah parameter diagnostik yang kritis dalam memahami seberapa baik baterai berfungsi dan seberapa parah degradasinya. Penurunan kapasitas dan peningkatan resistansi (baik internal maupun transfer muatan) adalah tanda-tanda klasik dari baterai yang menua atau rusak. Dengan menggunakan ketiga fitur ini, kita mendapatkan gambaran yang komprehensif tentang kesehatan baterai, memungkinkan kita mengkategorikannya menjadi 'aman', 'perlu di test lebih lanjut', atau 'tidak aman'.

#### Peran Potensial `ambient_temperature` sebagai Fitur Tambahan

Meskipun `Capacity`, `Re`, dan `Rct` adalah inti, `ambient_temperature` memiliki peran potensial yang signifikan sebagai fitur tambahan. Suhu lingkungan dapat sangat mempengaruhi kinerja baterai. Suhu ekstrem (terlalu panas atau terlalu dingin) dapat mempercepat degradasi, mengurangi kapasitas efektif, dan meningkatkan resistansi internal. Dalam model AI yang lebih canggih, memasukkan `ambient_temperature` dapat membantu model memahami konteks operasional baterai dan memprediksi degradasinya dengan lebih akurat. Misalnya, baterai yang menunjukkan resistansi sedikit tinggi pada suhu lingkungan sangat rendah mungkin masih dianggap 'aman' karena efek suhu, sementara resistansi yang sama pada suhu kamar mungkin menunjukkan masalah yang lebih serius.

#### Pentingnya `voltage` dan `current` (Input di Masa Depan)

Meskipun `voltage` dan `current` tidak tersedia dalam DataFrame `df` kita saat ini, keduanya adalah parameter yang sangat penting dalam analisis baterai secara umum. Mereka adalah dasar dari setiap pengukuran kinerja baterai:

*   **Voltage (Tegangan)**: Menunjukkan perbedaan potensial listrik antara dua terminal baterai. Tegangan baterai berubah selama siklus pengisian dan pengosongan, dan pola tegangan dapat memberikan wawasan tentang status pengisian (SoC), status kesehatan (SoH), dan potensi masalah internal (misalnya, _voltage sag_ yang tidak normal). Tegangan adalah salah satu parameter paling dasar yang digunakan untuk memantau baterai.
*   **Current (Arus)**: Mengukur laju aliran muatan listrik masuk atau keluar dari baterai. Arus sangat penting untuk memahami beban yang diberikan pada baterai, seberapa cepat ia mengisi atau mengosongkan, dan bagaimana hal itu mempengaruhi kapasitas dan resistansinya dari waktu ke waktu. Arus berlebih dapat menyebabkan pemanasan dan degradasi yang cepat.

Jika data `voltage` dan `current` dapat diperoleh di masa depan (misalnya, dari sistem manajemen baterai yang lebih lengkap atau sensor tambahan), mereka dapat diintegrasikan sebagai fitur input yang sangat berharga untuk model AI. Data ini akan memungkinkan model untuk:

1.  **Mendeteksi Anomali Real-time**: Perubahan mendadak atau pola abnormal dalam tegangan dan arus dapat menjadi indikasi awal kegagalan atau degradasi.
2.  **Memperkirakan SoC dan SoH Lebih Akurat**: Dengan tegangan dan arus, model dapat lebih akurat memperkirakan status pengisian dan kesehatan baterai.
3.  **Memahami Perilaku Dinamis**: `voltage` dan `current` memungkinkan analisis perilaku baterai di bawah berbagai kondisi beban, yang penting untuk aplikasi yang menuntut.
4.  **Meningkatkan Prediksi Umur Baterai**: Model dapat dilatih untuk mengidentifikasi bagaimana pola penggunaan (yang direfleksikan oleh arus dan tegangan) memengaruhi umur baterai dalam jangka panjang.

Dengan demikian, penambahan `voltage` dan `current` akan memberikan model AI pemahaman yang lebih dinamis dan holistik tentang kondisi operasional dan kesehatan baterai.

## Final Task

### Subtask:
Menyimpulkan diskusi mengenai fitur input yang paling sesuai untuk model AI saat ini dan potensi fitur tambahan untuk pengembangan di masa mendatang, serta bagaimana informasi ini disampaikan ke tim front-end.


# Task
Menyimpulkan diskusi mengenai fitur input yang paling sesuai untuk model AI saat ini, yaitu `Capacity`, `Re`, `Rct`, dan `ambient_temperature`. Juga akan dibahas potensi fitur tambahan seperti `voltage` dan `current` untuk pengembangan di masa mendatang. Selain itu, akan disebutkan bagaimana model AI dapat menggunakan metode *Nearest Neighbor* untuk klasifikasi status baterai, dan bagaimana semua informasi ini (input fitur, status klasifikasi, dan rekomendasi) akan disampaikan secara efektif kepada tim front-end untuk implementasi.

## Summary:

### Q&A
**1. Which input features are most suitable for the current AI model?**
The most suitable input features for the current AI model are `Capacity`, `Re` (Resistance), `Rct` (Resistance to Charge Transfer), and potentially `ambient_temperature` as an additional contextual feature.

**2. What are the potential additional features for future development?**
For future development, `voltage` and `current` are crucial potential additional features, as they provide real-time dynamic behavior insights into the battery's performance and health.

**3. How can the AI model use classification for battery status?**
The AI model can use a classification method, such as *Nearest Neighbor* (as suggested in the initial task, though not explicitly implemented in the code steps), to categorize battery status into 'aman' (safe), 'perlu di test lebih lanjut' (needs further testing), or 'tidak aman' (unsafe) based on the input features.

**4. How will this information be effectively conveyed to the front-end team for implementation?**
All information (input features, classification status, and recommendations) will be conveyed to the front-end team through a structured API. The front-end will display the `Battery_Status` using intuitive indicators (e.g., text, color codes), and provide actionable recommendations.

### Data Analysis Key Findings
*   **Key Input Features Identified**: The analysis confirmed `Capacity`, `Re`, and `Rct` as critical features for assessing battery health due to their direct reflection of internal battery conditions. `ambient_temperature` was also identified as a valuable contextual feature.
*   **Descriptive Statistics for Classification**: Descriptive statistics for `Capacity`, `Re`, and `Rct` were calculated and used to define thresholds for battery classification:
    *   **Capacity**: Mean = 0.8249, 25th percentile = 0.7751, 75th percentile = 0.9865.
    *   **Re**: Mean = 0.0777, 75th percentile = 0.0958.
    *   **Rct**: Mean = 0.1251, 75th percentile = 0.1589.
*   **Battery Status Classification**: A new column, `Battery_Status`, was successfully created in the DataFrame `df` based on the defined thresholds, categorizing batteries into 'aman', 'perlu di test lebih lanjut', and 'tidak aman'.
    *   `tidak aman`: 3920 instances
    *   `aman`: 2847 instances
    *   `perlu di test lebih lanjut`: 601 instances
*   **Visualized Distribution**: The distribution of battery statuses was visualized using a bar plot, clearly showing the proportion of batteries in each category.
*   **Future Feature Importance**: `voltage` and `current`, although not present in the current dataset, were highlighted as highly important parameters for future AI model enhancements, enabling more dynamic and accurate battery health monitoring and anomaly detection.

### Insights or Next Steps
*   **Refine Classification Thresholds**: The initial thresholds were based on descriptive statistics (mean, percentiles). For a production-ready system, these thresholds should be refined through expert knowledge, domain-specific requirements, and potentially more advanced statistical methods or machine learning approaches to optimize classification accuracy.
*   **Integrate Dynamic Features**: Prioritize the acquisition and integration of `voltage` and `current` data. This will allow for the development of more sophisticated AI models capable of real-time anomaly detection, more accurate State of Charge (SoC) and State of Health (SoH) estimation, and predictive maintenance capabilities for batteries.


# Task
Menganalisis data baterai, mendefinisikan kriteria klasifikasi berdasarkan statistik deskriptif dari 'Capacity', 'Re', dan 'Rct', membuat kolom status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman'), memvisualisasikan distribusi status baterai, dan menjelaskan penggunaan fitur serta output untuk AI/front-end.

## Prepare Data for Nearest Neighbor

### Subtask:
Pilih fitur `Capacity`, `Re`, `Rct`, dan `ambient_temperature` serta target `Battery_Status` dari DataFrame. Bagi data menjadi set pelatihan dan pengujian, dan lakukan penskalaan fitur jika diperlukan untuk algoritma Nearest Neighbor.


**Reasoning**:
I will select the specified feature columns (`Capacity`, `Re`, `Rct`, `ambient_temperature`) and the target column (`Battery_Status`) from the DataFrame `df`.



In [ ]:
X = df[['Capacity', 'Re', 'Rct', 'ambient_temperature']]
y = df['Battery_Status']

print("Features (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

**Reasoning**:
I will split the data into training and testing sets using `train_test_split` from `sklearn.model_selection` and then apply `StandardScaler` from `sklearn.preprocessing` to scale the features in both the training and testing sets. This is crucial for distance-based algorithms like Nearest Neighbor.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Shape of X_train_scaled:", X_train_scaled.shape)
print("Shape of X_test_scaled:", X_test_scaled.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

### Menyeimbangkan Data Menggunakan SMOTE

Karena dataset kita tidak seimbang (jumlah sampel untuk 'perlu di test lebih lanjut' jauh lebih sedikit dibandingkan kelas lain), kita akan menggunakan *Synthetic Minority Over-sampling Technique* (SMOTE) untuk menyeimbangkan kelas-kelas minoritas dalam data pelatihan. Ini akan membantu model Nearest Neighbor agar tidak bias terhadap kelas mayoritas.

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Distribusi kelas sebelum SMOTE: {Counter(y_train)}")

sm = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

print(f"Distribusi kelas setelah SMOTE: {Counter(y_train_resampled)}")

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier

# Jalur file yang benar berdasarkan ekstraksi di sel SCWniC5IfhZL
file_path = 'nasa_battery_data/Battery_Data_Cleaned.csv'

# Fallback check
if not os.path.exists(file_path):
    file_path = 'Battery_Data_Cleaned.csv'

# 1. Reload data
try:
    df = pd.read_csv(file_path)
    print(f"File berhasil dimuat dari: {file_path}")
except FileNotFoundError:
    print("File tidak ditemukan. Pastikan sel ekstraksi (SCWniC5IfhZL) sudah dijalankan.")
    raise

# Re-apply thresholds to create Battery_Status
conditions = [
    (df['Capacity'] > 0.8249) & (df['Re'] < 0.0777) & (df['Rct'] < 0.1251),
    (df['Capacity'] < 0.7751) | (df['Re'] > 0.0958) | (df['Rct'] > 0.1589)
]
choices = ['aman', 'tidak aman']
df['Battery_Status'] = np.select(conditions, choices, default='perlu di test lebih lanjut')

# 2. Prepare Features and Target
X = df[['Capacity', 'Re', 'Rct', 'ambient_temperature']]
y = df['Battery_Status']

# 3. Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Apply SMOTE
sm = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

# 5. Train KNN on Balanced Data
knn_resampled_model = KNeighborsClassifier(n_neighbors=5)
knn_resampled_model.fit(X_train_resampled, y_train_resampled)
y_pred_resampled = knn_resampled_model.predict(X_test_scaled)

print("Model KNN (SMOTE) berhasil dilatih.")
print(f"Distribusi kelas setelah penyeimbangan: {dict(pd.Series(y_train_resampled).value_counts())}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate the model
print("Classification Report (Trained on Balanced Data):")
print(classification_report(y_test, y_pred_resampled))

# Visualization of Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_resampled)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=knn_resampled_model.classes_,
            yticklabels=knn_resampled_model.classes_)
plt.title('Confusion Matrix - Balanced Model')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### Alternatif: Menggunakan Pembobotan Jarak (Weighted KNN)
Sebagai ganti SMOTE, kita bisa menggunakan parameter `weights='distance'` agar tetangga yang lebih dekat memiliki pengaruh lebih besar. Ini sangat efektif jika data tidak seimbang tetapi memiliki pola spasial yang jelas.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# 1. Load & Preprocess Data
file_path = 'nasa_battery_data/Battery_Data_Cleaned.csv'
if not os.path.exists(file_path): file_path = 'Battery_Data_Cleaned.csv'

try:
    df = pd.read_csv(file_path)
    # Re-apply thresholds
    conditions = [
        (df['Capacity'] > 0.8249) & (df['Re'] < 0.0777) & (df['Rct'] < 0.1251),
        (df['Capacity'] < 0.7751) | (df['Re'] > 0.0958) | (df['Rct'] > 0.1589)
    ]
    choices = ['aman', 'tidak aman']
    df['Battery_Status'] = np.select(conditions, choices, default='perlu di test lebih lanjut')

    X = df[['Capacity', 'Re', 'Rct', 'ambient_temperature']]
    y = df['Battery_Status']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 2. Pendekatan A: SMOTE (Over-sampling)
    sm = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

    knn_smote = KNeighborsClassifier(n_neighbors=5)
    knn_smote.fit(X_train_resampled, y_train_resampled)

    # 3. Pendekatan B: Weighted KNN (Penyeimbangan via Bobot Jarak)
    knn_weighted = KNeighborsClassifier(n_neighbors=5, weights='distance')
    knn_weighted.fit(X_train_scaled, y_train)

    print("--- Distribusi Kelas Training (Setelah SMOTE) ---")
    print(pd.Series(y_train_resampled).value_counts())

    print("\n--- Evaluasi KNN + SMOTE ---")
    print(classification_report(y_test, knn_smote.predict(X_test_scaled)))

    print("\n--- Evaluasi Weighted KNN (Tanpa SMOTE) ---")
    print(classification_report(y_test, knn_weighted.predict(X_test_scaled)))

except Exception as e:
    print(f"Error: {e}. Pastikan file dataset sudah diekstrak.")

### Pendekatan Alternatif: Class Weights
Sebagai alternatif dari SMOTE (yang menciptakan data sintetis), kita bisa menggunakan parameter `class_weight='balanced'`. Metode ini memberikan penalti yang lebih besar untuk kesalahan pada kelas minoritas selama pelatihan, sehingga model lebih sensitif terhadap label yang jarang muncul.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# 1. Pemuatan Data dan Preprocessing
file_path = 'nasa_battery_data/Battery_Data_Cleaned.csv'
if not os.path.exists(file_path): file_path = 'Battery_Data_Cleaned.csv'

try:
    df = pd.read_csv(file_path)
    # Definisikan ulang status berdasarkan ambang batas
    conditions = [
        (df['Capacity'] > 0.8249) & (df['Re'] < 0.0777) & (df['Rct'] < 0.1251),
        (df['Capacity'] < 0.7751) | (df['Re'] > 0.0958) | (df['Rct'] > 0.1589)
    ]
    choices = ['aman', 'tidak aman']
    df['Battery_Status'] = np.select(conditions, choices, default='perlu di test lebih lanjut')

    X = df[['Capacity', 'Re', 'Rct', 'ambient_temperature']]
    y = df['Battery_Status']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 2. Inisialisasi Model dengan Class Weight 'balanced'
    # Ini akan secara otomatis menyesuaikan bobot berdasarkan frekuensi kelas
    rf_weighted = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf_weighted.fit(X_train_scaled, y_train)

    # 3. Prediksi dan Evaluasi
    y_pred_weighted = rf_weighted.predict(X_test_scaled)

    print("--- Laporan Klasifikasi (Pendekatan Class Weights) ---")
    print(classification_report(y_test, y_pred_weighted))

    # Visualisasi Confusion Matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred_weighted)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=rf_weighted.classes_,
                yticklabels=rf_weighted.classes_)
    plt.title('Confusion Matrix - Class Weights Approach')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

except Exception as e:
    print(f"Error: {e}. Pastikan Anda telah menjalankan sel pengunduhan data (SCWniC5IfhZL) terlebih dahulu.")

### Prediksi Menggunakan Model Class Weights

Sekarang kita akan menggunakan model `rf_weighted` yang telah dilatih untuk memprediksi status dari data baterai baru. Ingat bahwa data baru harus melalui proses `scaler.transform` yang sama sebelum dimasukkan ke dalam model.

In [ ]:
# 1. Siapkan data contoh baru
contoh_baterai = pd.DataFrame({
    'Capacity': [0.95, 0.81, 0.65],
    'Re': [0.06, 0.08, 0.12],
    'Rct': [0.10, 0.14, 0.20],
    'ambient_temperature': [25, 25, 25]
})

# 2. Penskalaan data baru
contoh_scaled = scaler.transform(contoh_baterai)

# 3. Prediksi menggunakan Random Forest (Class Weights)
prediksi = rf_weighted.predict(contoh_scaled)
probabilitas = rf_weighted.predict_proba(contoh_scaled)

# 4. Tampilkan Hasil
for i in range(len(prediksi)):
    print(f"Baterai {i+1}:")
    print(f"  - Prediksi Status: {prediksi[i].upper()}")
    print(f"  - Keyakinan Model: {max(probabilitas[i])*100:.2f}%")
    print("-" * 30)

### Persiapan Export Model ke JavaScript

Untuk menjalankan model ini di browser (TensorFlow.js/Vanilla JS), kita perlu mengekspor dua hal:
1. **Parameter Scaler**: Agar data input di front-end memiliki skala yang sama dengan saat training.
2. **Struktur Random Forest**: Karena RF bukan Neural Network, kita akan menyimpannya sebagai file JSON yang merepresentasikan kumpulan *Decision Trees*.

In [ ]:
import json
import joblib

# 1. Simpan Scaler (Mean & Scale) untuk digunakan di Front-end
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'features': ['Capacity', 'Re', 'Rct', 'ambient_temperature']
}

with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)

# 2. Simpan model Random Forest (Format .joblib sebagai cadangan backend)
joblib.dump(rf_weighted, 'random_forest_model.joblib')

print("File 'scaler_params.json' dan 'random_forest_model.joblib' berhasil dibuat!")

### Opsi Implementasi di Front-end

Anda memiliki dua pilihan utama untuk menggunakan model ini di JavaScript:

1.  **Library `m2cgen` (Model to Code Generator)**: Mengonversi model menjadi kode JavaScript murni (Tanpa dependensi).
2.  **Library `scikit-learn-js`**: Pustaka JavaScript yang bisa memuat file model JSON dari scikit-learn.

Berikut adalah contoh cara mengonversi model menjadi kode JavaScript murni menggunakan `m2cgen`:

In [ ]:
!pip install -q m2cgen
import m2cgen as m2c

# Konversi model Random Forest ke kode JavaScript murni
js_code = m2c.export_to_javascript(rf_weighted)

with open('model_logic.js', 'w') as f:
    f.write(js_code)

print("File 'model_logic.js' berhasil dibuat! Anda bisa langsung memanggil fungsi score() di JavaScript.")

### Serving Model via FastAPI (Backend API)

Jika Anda ingin model ini berjalan di server dan diakses oleh web/mobile via API, Anda bisa menggunakan **FastAPI**. Berikut adalah kerangka kodenya:

In [ ]:
!pip install -q fastapi uvicorn pyngrok

# Kode ini adalah contoh skrip main.py yang akan dijalankan di server
serving_script = """
from fastapi import FastAPI
import joblib
import numpy as np
from pydantic import BaseModel

app = FastAPI()

# Load model dan scaler
model = joblib.load('random_forest_model.joblib')
# Catatan: Scaler idealnya juga disimpan via joblib untuk kemudahan

class BatteryInput(BaseModel):
    capacity: float
    re: float
    rct: float
    temp: float

@app.post("/predict")
def predict_status(data: BatteryInput):
    # Urutan fitur: Capacity, Re, Rct, ambient_temperature
    features = np.array([[data.capacity, data.re, data.rct, data.temp]])

    # Lakukan prediksi
    prediction = model.predict(features)
    return {"status": prediction[0]}
"""

with open('server.py', 'w') as f:
    f.write(serving_script)

print("Skrip server.py siap. Di produksi, Anda cukup menjalankan: uvicorn server:app")

### Implementasi Serving FastAPI yang Siap Pakai

Kita akan menggunakan `nest_asyncio` agar server FastAPI bisa berjalan di dalam environment Colab, dan `pyngrok` untuk membuat terowongan (tunnel) publik agar API bisa diakses oleh front-end Anda.

In [ ]:
!pip install -q fastapi uvicorn nest_asyncio pyngrok

import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import numpy as np
import json
from fastapi.middleware.cors import CORSMiddleware

# 1. Inisialisasi FastAPI
app = FastAPI(title="Battery Status Prediction API")

# Aktifkan CORS agar bisa dipanggil dari web front-end (React/Vue/HTML)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. Load Model dan Scaler Params
model = joblib.load('random_forest_model.joblib')
with open('scaler_params.json', 'r') as f:
    scaler_data = json.load(f)

mean = np.array(scaler_data['mean'])
scale = np.array(scaler_data['scale'])

# 3. Skema Data Input
class BatteryData(BaseModel):
    capacity: float
    re: float
    rct: float
    ambient_temperature: float

# 4. Endpoint Prediksi
@app.post("/predict")
def predict(data: BatteryData):
    try:
        # Susun input dan lakukan manual scaling (karena kita hanya simpan mean/scale)
        raw_features = np.array([[data.capacity, data.re, data.rct, data.ambient_temperature]])
        scaled_features = (raw_features - mean) / scale

        # Prediksi
        prediction = model.predict(scaled_features)[0]
        probabilities = model.predict_proba(scaled_features)[0]

        # Ambil probabilitas tertinggi sebagai tingkat keyakinan
        confidence = float(max(probabilities))

        return {
            "status": prediction,
            "confidence": confidence,
            "input_received": data.dict()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
def root():
    return {"message": "API Status: Online"}

print("Aplikasi FastAPI telah dikonfigurasi.")

### Menjalankan Server

Jalankan kode di bawah ini untuk mengaktifkan server. Jika Anda memiliki `authtoken` dari Ngrok, Anda bisa menggunakannya untuk mendapatkan URL publik tetap.

In [ ]:
import uvicorn
import asyncio

# Memastikan nest_asyncio sudah diaktifkan
nest_asyncio.apply()

async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

# Menjalankan server sebagai background task di event loop yang sudah ada
loop = asyncio.get_event_loop()
if loop.is_running():
    loop.create_task(run_server())
    print("Server sedang berjalan di latar belakang: http://localhost:8000")
    print("Anda bisa membuka http://localhost:8000/docs untuk dokumentasi Swagger API.")
else:
    asyncio.run(run_server())

### Ringkasan Pilihan Serving:

*   **model_logic.js**: Langsung pasang di folder proyek web Anda (Sangat Cepat).
*   **FastAPI**: Gunakan jika data baterai dikirim dari database atau butuh keamanan tambahan.

### Perbandingan Performa: Random Forest vs. Weighted KNN

Bagian ini membandingkan hasil evaluasi dari kedua pendekatan penyeimbangan data (bobot kelas vs bobot jarak) untuk melihat efektivitasnya dalam mendeteksi kelas minoritas.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Ekstrak metrik untuk Random Forest
rf_metrics = precision_recall_fscore_support(y_test, y_pred_weighted, average='macro')

# 2. Ekstrak metrik untuk Weighted KNN
# Pastikan y_pred dari weighted KNN tersedia
y_pred_knn_w = knn_weighted.predict(X_test_scaled)
knn_metrics = precision_recall_fscore_support(y_test, y_pred_knn_w, average='macro')

# 3. Buat DataFrame untuk visualisasi
comparison_df = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score'],
    'Random Forest (Class Weights)': rf_metrics[:3],
    'Weighted KNN': knn_metrics[:3]
})

comparison_melted = comparison_df.melt(id_vars='Metric', var_name='Model', value_name='Score')

# 4. Visualisasi
plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_melted, x='Metric', y='Score', hue='Model', palette='muted')
plt.ylim(0.9, 1.01) # Zoom in pada area performa tinggi
plt.title('Macro Average Performance Comparison')
plt.ylabel('Score (Macro Avg)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

display(comparison_df)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# Now X_train_scaled and y_train are guaranteed to be defined by cell 69cddaed
knn_weighted = KNeighborsClassifier(n_neighbors=5, weights='distance')

# Train on original scaled data (Weighted approach)
knn_weighted.fit(X_train_scaled, y_train)

# Predict
y_pred_weighted = knn_weighted.predict(X_test_scaled)

print("Classification Report (Weighted KNN - Tanpa SMOTE):")
print(classification_report(y_test, y_pred_weighted))

As you can see, before SMOTE, the class 'perlu di test lebih lanjut' was the minority class with 481 samples. After applying SMOTE, all three classes ('tidak aman', 'aman', 'perlu di test lebih lanjut') now have 3136 samples in the training set, making the data balanced for training the Nearest Neighbor model.

## Implement Nearest Neighbor Model

### Subtask:
Latih model klasifikasi Nearest Neighbor (misalnya, menggunakan `KNeighborsClassifier` dari scikit-learn) pada data pelatihan yang telah disiapkan.


**Reasoning**:
I need to import `KNeighborsClassifier`, initialize it, train it on the scaled training data, and make predictions on the scaled test data.



In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Instantiate KNeighborsClassifier with n_neighbors=5
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = knn_model.predict(X_test_scaled)

print("KNeighborsClassifier model trained and predictions made.")

## Evaluate and Explain Results

### Subtask:
Evaluasi kinerja model Nearest Neighbor menggunakan metrik yang relevan (misalnya, akurasi, presisi, recall, F1-score) dan jelaskan bagaimana model ini membuat prediksi status baterai berdasarkan 4 fitur input.


**Reasoning**:
To evaluate the performance of the trained Nearest Neighbor model, I need to calculate standard classification metrics. I will use `classification_report` to get precision, recall, and F1-score, and `confusion_matrix` to understand the types of errors made by the classifier.



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

```markdown
### Analisis Kinerja Model Nearest Neighbor dan Penjelasan Prediksi

**1. Evaluasi Kinerja Model (Classification Report dan Confusion Matrix):**

Dari **Classification Report**:
*   **Akurasi Keseluruhan (Accuracy):** Model mencapai akurasi sebesar 1.00 (100%), yang mengindikasikan model memprediksi sebagian besar sampel dengan benar.
*   **Presisi (Precision):**
    *   `aman`: 1.00 (100%) - Dari semua prediksi 'aman', 100% benar-benar 'aman'.
    *   `perlu di test lebih lanjut`: 0.98 (98%) - Dari semua prediksi 'perlu di test lebih lanjut', 98% benar-benar 'perlu di test lebih lanjut'.
    *   `tidak aman`: 1.00 (100%) - Dari semua prediksi 'tidak aman', 100% benar-benar 'tidak aman'.
*   **Recall (Sensitivitas):**
    *   `aman`: 1.00 (100%) - Model berhasil mengidentifikasi 100% dari semua baterai yang sebenarnya 'aman'.
    *   `perlu di test lebih lanjut`: 0.97 (97%) - Model berhasil mengidentifikasi 97% dari semua baterai yang sebenarnya 'perlu di test lebih lanjut'.
    *   `tidak aman`: 1.00 (100%) - Model berhasil mengidentifikasi 100% dari semua baterai yang sebenarnya 'tidak aman'.
*   **F1-Score:** Merupakan rata-rata harmonis dari presisi dan recall. F1-score yang tinggi untuk semua kelas (1.00, 0.97, 1.00) menunjukkan keseimbangan yang baik antara presisi dan recall.

Dari **Confusion Matrix**:
```
[[568   2   0]
 [  1 116   3]
 [  1   0 783]]
```
*   **Baris 0 (`aman`):**
    *   568 sampel 'aman' diprediksi dengan benar sebagai 'aman'.
    *   2 sampel 'aman' salah diprediksi sebagai 'perlu di test lebih lanjut'.
*   **Baris 1 (`perlu di test lebih lanjut`):**
    *   1 sampel 'perlu di test lebih lanjut' salah diprediksi sebagai 'aman'.
    *   116 sampel 'perlu di test lebih lanjut' diprediksi dengan benar.
    *   3 sampel 'perlu di test lebih lanjut' salah diprediksi sebagai 'tidak aman'.
*   **Baris 2 (`tidak aman`):**
    *   1 sampel 'tidak aman' salah diprediksi sebagai 'aman'.
    *   783 sampel 'tidak aman' diprediksi dengan benar sebagai 'tidak aman'.

Secara keseluruhan, model Nearest Neighbor menunjukkan kinerja yang sangat baik dengan akurasi dan F1-score yang tinggi di semua kelas. Jumlah misklasifikasi sangat rendah, menunjukkan bahwa fitur yang dipilih dan ambang batas yang ditetapkan cukup efektif untuk membedakan status baterai.

**2. Bagaimana Model Nearest Neighbor Membuat Prediksi Berdasarkan 4 Fitur Input:**

Model K-Nearest Neighbors (KNN) adalah algoritma berbasis instansi (instance-based) dan non-parametrik. Cara kerjanya adalah sebagai berikut:

1.  **Penyimpanan Data Pelatihan:** Selama fase pelatihan (`knn_model.fit`), model KNN hanya menyimpan semua data pelatihan yang tersedia (`X_train_scaled` dan `y_train`). Tidak ada model "belajar" dalam artian membangun fungsi matematis yang kompleks.

2.  **Perhitungan Jarak (Distance Calculation):** Ketika model menerima sebuah sampel baru (misalnya, sebuah baterai dengan nilai `Capacity`, `Re`, `Rct`, dan `ambient_temperature` yang baru) untuk diprediksi, model akan menghitung "jarak" antara sampel baru ini dengan *setiap* sampel di dalam data pelatihan yang sudah disimpan. Jarak ini biasanya dihitung menggunakan metrik seperti jarak Euclidean atau Manhattan. Karena fitur telah di-skalakan (`StandardScaler`), semua fitur berkontribusi secara proporsional dalam perhitungan jarak.

3.  **Identifikasi Tetangga Terdekat:** Setelah menghitung semua jarak, model akan mengidentifikasi `k` tetangga terdekat (dalam kasus ini, `k=5`) dari sampel baru tersebut di antara seluruh data pelatihan. "Terdekat" di sini berarti memiliki jarak terkecil.

4.  **Prediksi Kelas (Voting):** Untuk mengklasifikasikan sampel baru, model akan melihat label kelas (status baterai: 'aman', 'perlu di test lebih lanjut', 'tidak aman') dari `k` tetangga terdekat tersebut. Kelas yang paling sering muncul di antara `k` tetangga ini akan menjadi prediksi untuk sampel baru tersebut (prinsip "majority vote").

**Contoh:**
Misalkan kita memiliki baterai baru dengan `Capacity`, `Re`, `Rct`, dan `ambient_temperature` tertentu. Model akan mencari 5 baterai yang paling mirip dari data latihannya. Jika 4 dari 5 baterai terdekat tersebut berlabel 'aman' dan 1 berlabel 'perlu di test lebih lanjut', maka model akan memprediksi baterai baru tersebut sebagai 'aman'.

Dengan demikian, model KNN memprediksi status baterai berdasarkan kesamaan dengan baterai-baterai yang sudah dikenal sebelumnya, menggunakan keempat fitur (`Capacity`, `Re`, `Rct`, `ambient_temperature`) untuk mengukur kesamaan tersebut.

## Final Task

### Subtask:
Merangkum implementasi dan hasil klasifikasi Nearest Neighbor, serta bagaimana informasi ini dapat digunakan lebih lanjut dalam konteks proyek Anda.


# Task
Menganalisis data baterai, mendefinisikan kriteria klasifikasi berdasarkan statistik deskriptif dari 'Capacity', 'Re', dan 'Rct', membuat kolom status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman'), memvisualisasikan distribusi status baterai, dan menjelaskan penggunaan fitur serta output untuk AI/front-end.

### Ringkasan Implementasi Model Nearest Neighbor dan Penggunaan Informasi Lebih Lanjut

**1. Implementasi dan Kinerja Model Nearest Neighbor:**

Kita telah berhasil mengimplementasikan model klasifikasi **K-Nearest Neighbors (KNN)** untuk memprediksi status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman') berdasarkan empat fitur utama: `Capacity`, `Re`, `Rct`, dan `ambient_temperature`. Proses implementasi meliputi:

*   **Pemilihan Fitur dan Target**: Fitur input (`Capacity`, `Re`, `Rct`, `ambient_temperature`) dan target (`Battery_Status`) diekstrak dari DataFrame.
*   **Pembagian Data**: Data dibagi menjadi set pelatihan (80%) dan pengujian (20%) dengan `stratify=y` untuk memastikan distribusi kelas yang seimbang di kedua set.
*   **Penskalaan Fitur**: `StandardScaler` digunakan untuk menormalisasi fitur input. Langkah ini sangat penting untuk algoritma berbasis jarak seperti KNN agar semua fitur berkontribusi secara adil dalam perhitungan jarak.
*   **Pelatihan Model**: Model `KNeighborsClassifier` dilatih pada data pelatihan yang telah diskalakan dengan `n_neighbors=5`.
*   **Prediksi**: Model membuat prediksi pada data pengujian yang telah diskalakan.

**Evaluasi Kinerja:**

Evaluasi model menunjukkan kinerja yang **sangat baik**, dengan akurasi keseluruhan mendekati 100% dan F1-score yang tinggi untuk semua kelas ('aman', 'perlu di test lebih lanjut', 'tidak aman'). Confusion matrix mengkonfirmasi bahwa hanya ada sejumlah kecil misklasifikasi. Ini menunjukkan bahwa kriteria klasifikasi yang kita definisikan dan fitur yang dipilih sangat efektif dalam membedakan status baterai.

**2. Bagaimana Informasi Ini Dapat Digunakan Lebih Lanjut dalam Konteks Proyek:**

Keberhasilan model KNN dalam mengklasifikasi status baterai membuka banyak peluang untuk pengembangan lebih lanjut dalam proyek ini:

*   **Sistem Pemantauan Kesehatan Baterai Real-time**: Model yang sudah dilatih dapat diintegrasikan ke dalam sistem pemantauan. Data sensor dari baterai (Capacity, Re, Rct, Ambient Temperature) dapat diumpankan ke model, dan status kesehatan baterai dapat ditampilkan secara real-time di front-end.

*   **Deteksi Anomali dan Peringatan Dini**: Karena model mampu mengidentifikasi baterai yang 'tidak aman' atau 'perlu di test lebih lanjut' dengan akurasi tinggi, sistem dapat memicu peringatan otomatis kepada pengguna atau operator untuk melakukan intervensi, seperti pemeriksaan lebih lanjut atau penggantian baterai.

*   **Dasar untuk Pengembangan Model AI Lanjutan**: Kinerja kuat dari model KNN ini dapat menjadi _baseline_ untuk membandingkan dengan algoritma klasifikasi lain (misalnya, Support Vector Machine, Random Forest, Neural Networks) yang mungkin dapat menangani pola yang lebih kompleks atau dataset yang lebih besar di masa depan.

*   **Rekomendasi Tindakan Preventif/Perbaikan**: Output klasifikasi dapat dipadukan dengan modul rekomendasi untuk memberikan saran tindakan yang spesifik. Misalnya, jika 'perlu di test lebih lanjut', sistem bisa menyarankan "Periksa koneksi terminal baterai" atau "Lakukan uji kapasitas penuh".

*   **Umpan Balik untuk Pengambilan Keputusan**: Data historis tentang klasifikasi baterai dapat digunakan untuk analisis lebih lanjut, seperti mengidentifikasi pola kegagalan umum, umur pakai rata-rata, atau efektivitas langkah-langkah pemeliharaan.

Dengan model yang terbukti andal ini, proyek Anda memiliki fondasi yang kuat untuk membangun solusi diagnostik baterai yang canggih dan bermanfaat.

## Kesiapan Model untuk Front-end dan Pertimbangan Implementasi

### Subtask:
Jelaskan kesiapan model Nearest Neighbor yang telah dilatih dan dievaluasi untuk integrasi ke front-end, serta membahas pertimbangan penting untuk implementasi di lingkungan produksi, seperti kebutuhan API dan validasi lebih lanjut.


```markdown
### Ringkasan Implementasi Model Nearest Neighbor dan Penggunaan Informasi Lebih Lanjut

**1. Implementasi dan Kinerja Model Nearest Neighbor:**

Kita telah berhasil mengimplementasikan model klasifikasi **K-Nearest Neighbors (KNN)** untuk memprediksi status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman') berdasarkan empat fitur utama: `Capacity`, `Re`, `Rct`, dan `ambient_temperature`. Proses implementasi meliputi:

*   **Pemilihan Fitur dan Target**: Fitur input (`Capacity`, `Re`, `Rct`, `ambient_temperature`) dan target (`Battery_Status`) diekstrak dari DataFrame.
*   **Pembagian Data**: Data dibagi menjadi set pelatihan (80%) dan pengujian (20%) dengan `stratify=y` untuk memastikan distribusi kelas yang seimbang di kedua set.
*   **Penskalaan Fitur**: `StandardScaler` digunakan untuk menormalisasi fitur input. Langkah ini sangat penting untuk algoritma berbasis jarak seperti KNN agar semua fitur berkontribusi secara adil dalam perhitungan jarak.
*   **Pelatihan Model**: Model `KNeighborsClassifier` dilatih pada data pelatihan yang telah diskalakan dengan `n_neighbors=5`.
*   **Prediksi**: Model membuat prediksi pada data pengujian yang telah diskalakan.

**Evaluasi Kinerja:**

Evaluasi model menunjukkan kinerja yang **sangat baik**, dengan akurasi keseluruhan mendekati 100% dan F1-score yang tinggi untuk semua kelas ('aman', 'perlu di test lebih lanjut', 'tidak aman'). Confusion matrix mengkonfirmasi bahwa hanya ada sejumlah kecil misklasifikasi. Ini menunjukkan bahwa kriteria klasifikasi yang kita definisikan dan fitur yang dipilih sangat efektif dalam membedakan status baterai.

**2. Bagaimana Informasi Ini Dapat Digunakan Lebih Lanjut dalam Konteks Proyek:**

Keberhasilan model KNN dalam mengklasifikasi status baterai membuka banyak peluang untuk pengembangan lebih lanjut dalam proyek ini:

*   **Sistem Pemantauan Kesehatan Baterai Real-time**: Model yang sudah dilatih dapat diintegrasikan ke dalam sistem pemantauan. Data sensor dari baterai (Capacity, Re, Rct, Ambient Temperature) dapat diumpankan ke model, dan status kesehatan baterai dapat ditampilkan secara real-time di front-end.

*   **Deteksi Anomali dan Peringatan Dini**: Karena model mampu mengidentifikasi baterai yang 'tidak aman' atau 'perlu di test lebih lanjut' dengan akurasi tinggi, sistem dapat memicu peringatan otomatis kepada pengguna atau operator untuk melakukan intervensi, seperti pemeriksaan lebih lanjut atau penggantian baterai.

*   **Dasar untuk Pengembangan Model AI Lanjutan**: Kinerja kuat dari model KNN ini dapat menjadi _baseline_ untuk membandingkan dengan algoritma klasifikasi lain (misalnya, Support Vector Machine, Random Forest, Neural Networks) yang mungkin dapat menangani pola yang lebih kompleks atau dataset yang lebih besar di masa depan.

*   **Rekomendasi Tindakan Preventif/Perbaikan**: Output klasifikasi dapat dipadukan dengan modul rekomendasi untuk memberikan saran tindakan yang spesifik. Misalnya, jika 'perlu di test lebih lanjut', sistem bisa menyarankan "Periksa koneksi terminal baterai" atau "Lakukan uji kapasitas penuh".

*   **Umpan Balik untuk Pengambilan Keputusan**: Data historis tentang klasifikasi baterai dapat digunakan untuk analisis lebih lanjut, seperti mengidentifikasi pola kegagalan umum, umur pakai rata-rata, atau efektivitas langkah-langkah pemeliharaan.

Dengan model yang terbukti andal ini, proyek Anda memiliki fondasi yang kuat untuk membangun solusi diagnostik baterai yang canggih dan bermanfaat.
```

## Demonstrasi Prediksi Model pada Data Baru

### Subtask:
Demonstrasikan cara menggunakan model Nearest Neighbor yang telah dilatih untuk memprediksi status baterai ('aman', 'perlu di test lebih lanjut', 'tidak aman') pada data input baru yang berisi `Capacity`, `Re`, `Rct`, dan `ambient_temperature`. Ini akan mencakup langkah-langkah untuk menyiapkan (scaling) data baru sebelum prediksi.


**Reasoning**:
I will create a new DataFrame with example battery data and then scale these new data points using the previously fitted `scaler` to prepare them for prediction by the trained `knn_model`.



In [ ]:
import pandas as pd

# 1. Create a new DataFrame with example input data
new_data = pd.DataFrame({
    'Capacity': [0.95, 0.70, 0.45, 1.05, 0.80],
    'Re': [0.06, 0.11, 0.15, 0.05, 0.08],
    'Rct': [0.10, 0.18, 0.22, 0.09, 0.14],
    'ambient_temperature': [25, 30, 15, 20, 35]
})

print("New Data (unscaled):")
display(new_data)

# 2. Preprocess the new data using the same StandardScaler
# Make sure to only use .transform() on the already fitted scaler
new_data_scaled = scaler.transform(new_data)

print("\nNew Data (scaled):")
display(pd.DataFrame(new_data_scaled, columns=new_data.columns))